# Ders 10 - Üretimde Yapay Zeka Ajanları

Bu derste Microsoft Agent Framework (Python) kullanarak yapay zeka ajanları için **üretim kalıplarını** öğreneceksiniz. Kapsanan konular:

- **Gözlemlenebilirlik** — ajan etkileşimlerine zamanlama ve günlükleme ekleme
- **Değerlendirme** — yanıt kalitesini puanlamak için bir değerlendirici ajan kullanma
- **Maliyet yönetimi** — token optimizasyonu ve model seçimi için stratejiler

Senaryo, kullanıcıların seyahat planlamasına yardımcı olan ve üzerine izleme ile değerlendirme eklenmiş bir **seyahat acentesi**.


## Kurulum


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Üretime Geçişte Dikkate Alınması Gerekenler

Yapay zeka ajanlarını prototiplerden üretime taşımak, üç temel alana dikkat etmeyi gerektirir:

1. **Gözlemlenebilirlik** — Ajanın ne yaptığını, ne kadar sürdüğünü ve hangi araçları çağırdığını görmeniz gerekir. İzleme ve günlükleme olmadan üretim sorunlarını ayıklamak neredeyse imkansızdır.

2. **Değerlendirme** — Otomatik kalite kontrolleri, ajan cevaplarının zaman içinde doğru, eksiksiz ve yararlı kalmasını sağlar. Bir değerlendirme ajanı, tanımlanmış ölçütlere göre cevapları puanlayabilir.

3. **Maliyet Yönetimi** — Token kullanımı doğrudan maliyeti etkiler. Prompt optimizasyonu, model seçimi ve önbelleğe alma gibi stratejiler, kaliteyi feda etmeden giderleri kontrol altında tutmaya yardımcı olur.


## Gözlemlenebilir Bir Ajan Oluşturma

Seyahat araçlarını tanımlar ve ajan çağrısını zamanlayarak sararız, böylece gecikmeyi izleyebiliriz. Üretimde OpenTelemetry veya benzeri bir izleme altyapısıyla entegre olurdunuz.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Değerlendirme Desenleri

Yaygın bir üretim deseni, ikinci bir ajanı **değerlendirici** olarak kullanmaktır. Değerlendirici, birincil ajanın yanıtını tamlık, doğruluk ve faydalılık gibi önceden tanımlanmış kriterlere göre puanlar.

Bu şunları sağlar:
- Yanıtlar kullanıcılara ulaşmadan önce otomatik kalite denetimleri
- İstemler veya modeller değiştiğinde regresyon tespiti
- Ajan performansının zaman içinde sürekli izlenmesi


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Maliyet Yönetimi Stratejileri

Maliyetleri kontrol etmek, üretim aşamasındaki yapay zeka ajanları için kritiktir. İşte başlıca stratejiler:

| Strateji | Açıklama |
|---|---|
| **Prompt optimizasyonu** | Sistem talimatlarını kısa tutun. Gereksiz bağlamı kaldırarak giriş token sayısını azaltın. |
| **Model seçimi** | Sınıflandırma veya çıkarım gibi basit görevler için daha küçük, daha ucuz modelleri (ör. GPT-4o-mini) kullanın ve daha karmaşık muhakeme gerektiren durumlar için daha büyük modelleri saklayın. |
| **Önbellekleme** | Alet sonuçlarını ve sık yapılan sorguları önbelleğe alarak gereksiz API çağrılarını önleyin. |
| **Token bütçeleri** | Beklenmedik derecede uzun yanıtları önlemek için `max_tokens` sınırları belirleyin. |
| **Toplu işleme** | Mümkün olduğunda birden fazla kullanıcı sorgusunu tek bir API çağrısında gruplandırın. |

Uygulamada, katmanlı bir yaklaşım iyi sonuç verir: basit talepleri hızlı ve ucuz bir modele yönlendirin ve yalnızca karmaşık sorguları daha yetenekli bir modele yükseltin.


## Özet

Bu derste şu konuları öğrendiniz:

1. **Gözlemlenebilirlik ekleyin** ajan etkileşimlerine zamanlama ve günlükleme ile, izleme ve iz sürme için zemin hazırlayarak.
2. **Ajan yanıtlarını değerlendirin** otomatik olarak, eksiksizlik, doğruluk ve yararlılık açısından puanlayan bir değerlendirici ajan kullanarak.
3. **Maliyetleri yönetin** prompt optimizasyonu, model seçimi, önbellekleme ve token bütçeleri aracılığıyla.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Feragatname:
Bu belge, yapay zeka çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba göstersek de, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayın. Orijinal belge, kendi dilindeki metni yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi tavsiye edilir. Bu çevirinin kullanılması sonucunda ortaya çıkabilecek herhangi bir yanlış anlama veya yanlış yorumdan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
